In [2]:
import sys
print(sys.executable)

c:\Users\tom39\Documents\.venv\.venv\Scripts\python.exe


In [4]:
import sys
import hydra
import torch
from omegaconf import OmegaConf
from pathlib import Path

print(sys.executable)
print("Hydra OK")
print("OmegaConf OK")

# Path to src folder (where mhnfs/ lives)
project_root = Path.cwd()
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

print(project_root)
print(project_root / "src")

c:\Users\tom39\Documents\.venv\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


c:\Users\tom39\Documents\.venv\.venv\Scripts\python.exe
Hydra OK
OmegaConf OK
c:\Users\tom39\Desktop\MHNfs
c:\Users\tom39\Desktop\MHNfs\src


In [5]:
from mhnfs.modules import ContextModule

c:\Users\tom39\Desktop\MHNfs\src\mhnfs\modules.py:679: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path="../configs", config_name="cfg")
c:\Users\tom39\Desktop\MHNfs\src\mhnfs\modules.py:698: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path="../configs", config_name="cfg")
c:\Users\tom39\Desktop\MHNfs\src\mhnfs\modules.py:719: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path="../configs", config_name="cfg")


# Minimal Input Value

In [6]:
torch.manual_seed(0)

# -----------------------------
# Minimal toy config (CONSISTENT)
# -----------------------------
cfg = OmegaConf.create({
    "system": {"ressources": {"device": "cpu"}},
    "model": {
        "associationSpace_dim": 4,
        "encoder": {
            "input_dim": 4,
            "activation": "relu",
            "number_hidden_layers": 0,
            "number_hidden_neurons": 8,
            "regularization": {
                "input_dropout": 0.0,
                "dropout": 0.0
            }
        },
        "layerNormBlock": {"usage": True, "affine": False},
        "transformer": {
            "activity_embedding_dim": 4,
            "num_layers": 1,
            "num_heads": 1,
            "dropout": 0.0,
            "dim_forward": 8,
            "activation": "relu"
        },
        "hopfield": {
            "dim_QK": 4,
            "heads": 1,
            "beta": 0.1,
            "dropout": 0.0
        }
    }
})

# -----------------------------
# Dummy tensors (MATCH config!)
# -----------------------------
B = 1
T_query = 2
T_support = 3
D = cfg.model.encoder.input_dim  # = 4

query = torch.randn(B, T_query, D)
support = torch.randn(B, T_support, D)
support_inactives_embedding = torch.zeros_like(support)
context_set_embedding = torch.zeros_like(query)

support_actives_mask = torch.ones(B, T_support, dtype=torch.bool)
support_inactives_mask = torch.ones(B, T_support, dtype=torch.bool)

# -----------------------------
# Test helper
# -----------------------------
def test_module(module_class, *inputs, cfg=None):
    module = module_class(cfg)
    print(f"\n=== {module_class.__name__} ===")
    for i, inp in enumerate(inputs):
        print(f"Input {i}: {inp.shape}")
    out = module(*inputs)
    if isinstance(out, tuple):
        for i, o in enumerate(out):
            print(f"Output {i}: {o.shape}")
    else:
        print(f"Output: {out.shape}")
    return out

In [7]:
import inspect
import mhnfs.modules as mod
from mhnfs.modules import ContextModule


classes = [name for name, obj in inspect.getmembers(mod, inspect.isclass)]
print("Classes in mhnfs.modules:", classes)

print(inspect.signature(ContextModule.__init__))

Classes in mhnfs.modules: ['ContextModule', 'CrossAttentionModule', 'EncoderBlock', 'Hopfield', 'LayerNormalizingBlock', 'OmegaConf', 'partial']
(self, cfg: omegaconf.omegaconf.OmegaConf)


In [8]:
# -----------------------------
# Imports
# -----------------------------
from mhnfs.modules import (
    ContextModule,
    EncoderBlock,
    LayerNormalizingBlock,
    CrossAttentionModule,
    Hopfield
)

# -----------------------------
# Run pipeline
# -----------------------------
q_ctx, *_ = test_module(
    ContextModule,
    query,
    support,
    support_inactives_embedding,
    context_set_embedding,
    cfg=cfg
)

q_enc = test_module(EncoderBlock, q_ctx, cfg=cfg)

q_ln, s_ln, si_ln = test_module(
    LayerNormalizingBlock,
    q_enc,
    support,
    support_inactives_embedding,
    cfg=cfg
)



=== ContextModule ===
Input 0: torch.Size([1, 2, 4])
Input 1: torch.Size([1, 3, 4])
Input 2: torch.Size([1, 3, 4])
Input 3: torch.Size([1, 2, 4])
Output 0: torch.Size([1, 1, 4])
Output 1: torch.Size([1, 3, 4])
Output 2: torch.Size([1, 4, 4])

=== EncoderBlock ===
Input 0: torch.Size([1, 1, 4])
Output: torch.Size([1, 1, 4])

=== LayerNormalizingBlock ===
Input 0: torch.Size([1, 1, 4])
Input 1: torch.Size([1, 3, 4])
Input 2: torch.Size([1, 3, 4])
Output 0: torch.Size([1, 1, 4])
Output 1: torch.Size([1, 3, 4])
Output 2: torch.Size([1, 3, 4])


Cross-Attension module

In [14]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from types import SimpleNamespace

# -----------------------------
# Utilities
# -----------------------------
def init_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

# -----------------------------
# GPT-like config
# -----------------------------
class GPTConfig:
    def __init__(self, n_embd, n_head=2):
        self.n_embd = n_embd
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        assert self.head_dim * n_head == n_embd

# -----------------------------
# RMSNorm
# -----------------------------
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(rms + self.eps)
        return x * self.weight

# -----------------------------
# Input embedding
# -----------------------------
class InputEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.type_emb = nn.Embedding(3, dim)

    def forward(self, x, types):
        return x + self.type_emb(types)

# -----------------------------
# SwiGLU
# -----------------------------
class SwiGLU(nn.Module):
    def forward(self, x):
        x, gate = x.chunk(2, dim=-1)
        return x * F.silu(gate)

In [15]:
# -----------------------------
# Mixture-of-Experts
# -----------------------------
class MoE(nn.Module):
    def __init__(self, d_model, n_experts=4, hidden_dim=32, dropout=0.05):
        super().__init__()
        self.router = nn.Linear(d_model, n_experts)
        self.n_experts = n_experts

        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, hidden_dim * 2),
                SwiGLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, d_model),
            ) for _ in range(n_experts)
        ])

    def forward(self, x, query=None, soft=True, support_count=None):
        B, T, D = x.shape

        n_experts_use = self.n_experts
        if support_count is not None:
            n_experts_use = min(self.n_experts, max(1, support_count // 2))

        experts = self.experts[:n_experts_use]
        logits = self.router(x)[:, :, :n_experts_use]

        if query is not None:
            q_vec = query.mean(dim=1, keepdim=True)
            sim = torch.einsum("bqd,btd->bqt", q_vec, x).squeeze(1)
            logits = logits + sim.unsqueeze(-1)

        probs = F.softmax(logits, dim=-1)

        if soft:
            out = sum(expert(x) * probs[..., i:i+1]
                      for i, expert in enumerate(experts))
        else:
            expert_id = logits.argmax(dim=-1)
            out = torch.zeros_like(x)
            for i, expert in enumerate(experts):
                mask = expert_id == i
                if mask.any():
                    out[mask] = expert(x[mask])

        return out

In [16]:
# -----------------------------
# Scale Dot Product Cross Attention
# -----------------------------
class ScaledDotProductCrossAttention(nn.Module):
    def __init__(self, config, attn_dropout=0.1, temp=1.0):
        super().__init__()

        self.n_head = config.n_head
        self.head_dim = config.head_dim
        self.attn_dropout = attn_dropout

        self.q_proj = nn.Linear(config.n_embd, config.n_embd)
        self.k_proj = nn.Linear(config.n_embd, config.n_embd)
        self.v_proj = nn.Linear(config.n_embd, config.n_embd)
        self.out_proj = nn.Linear(config.n_embd, config.n_embd)

        self.active_bias = nn.Embedding(1, config.n_embd)
        self.inactive_bias = nn.Embedding(1, config.n_embd)

        self.log_temp = nn.Parameter(torch.log(torch.tensor(temp)))

    def forward(self, q, kv, kv_mask=None, n_actives=None):
        B, Tq, D = q.shape
        Tk = kv.size(1)

        Na = int(n_actives)
        Ni = Tk - Na

        q = self.q_proj(q)

        kv = kv.clone()
        kv[:, :Na] += self.active_bias.weight
        kv[:, Na:] += self.inactive_bias.weight

        k = self.k_proj(kv)
        v = self.v_proj(kv)

        # reshape heads
        q = q.view(B, Tq, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, Tk, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, Tk, self.n_head, self.head_dim).transpose(1, 2)

        # stable scaling
        scale = 1.0 / math.sqrt(self.head_dim)
        q = q * scale

        temp = torch.exp(self.log_temp).clamp(0.1, 10)

        out = torch.zeros_like(q)

        if Na > 0:
            out_a = F.scaled_dot_product_attention(
                q / temp, k[:, :, :Na], v[:, :, :Na],
                dropout_p=self.attn_dropout
            )
            out += out_a / math.sqrt(max(Na, 1))

        if Ni > 0:
            out_i = F.scaled_dot_product_attention(
                q / temp, k[:, :, Na:], v[:, :, Na:],
                dropout_p=self.attn_dropout
            )
            out -= out_i / math.sqrt(max(Ni, 1))

        out = out.transpose(1, 2).contiguous().reshape(B, Tq, D)
        return self.out_proj(out)

In [17]:
# -----------------------------
# Transformer block
# -----------------------------
class TransformerBlock(nn.Module):
    def __init__(self, config, attn_dropout=0.1, update_supports=True):
        super().__init__()

        self.q_norm = RMSNorm(config.n_embd)
        self.kv_norm_a = RMSNorm(config.n_embd)
        self.kv_norm_i = RMSNorm(config.n_embd)

        self.cross_attn = ScaledDotProductCrossAttention(config, attn_dropout)
        self.delta_norm = RMSNorm(config.n_embd)

        self.ffn_norm = RMSNorm(config.n_embd)
        self.moe = MoE(config.n_embd)

        self.gate_attn = nn.Parameter(torch.tensor(0.3))
        self.gate_ffn = nn.Parameter(torch.tensor(0.7))

        self.update_supports = update_supports

    def forward(self, q, kv, kv_mask, n_actives):
        Na = n_actives

        kv_a = self.kv_norm_a(kv[:, :Na])
        kv_i = self.kv_norm_i(kv[:, Na:])
        kv_combined = torch.cat([kv_a, kv_i], dim=1)

        delta = self.cross_attn(self.q_norm(q), kv_combined, kv_mask, n_actives=Na)
        delta = self.delta_norm(delta)

        q = q + torch.sigmoid(self.gate_attn) * delta

        if self.update_supports:
            kv = kv + torch.sigmoid(self.gate_ffn) * self.moe(
                self.ffn_norm(kv),
                query=q,
                soft=self.training,
                support_count=kv.size(1)
            )

        return q, kv

In [18]:
# -----------------------------
# CrossAttentionModule
# -----------------------------
class CrossAttentionModule(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.model_dim = cfg.model.transformer.activity_embedding_dim
        config = GPTConfig(n_embd=self.model_dim)

        self.input_proj = nn.Linear(8, self.model_dim)
        self.embed = InputEmbedding(self.model_dim)
        self.block = TransformerBlock(config)

        self.apply(init_weights)

    def forward(self, query, actives, inactives, act_mask, inact_mask):
        B = query.size(0)
        D = query.size(-1)

        query_shape = query.shape

        query = query.view(B, -1, D)
        actives = actives.view(B, -1, D)
        inactives = inactives.view(B, -1, D)

        query = self.input_proj(query)
        actives = self.input_proj(actives)
        inactives = self.input_proj(inactives)

        n_actives = actives.size(1)

        kv = torch.cat([actives, inactives], dim=1)
        kv_mask = torch.cat([act_mask.view(B, -1), inact_mask.view(B, -1)], dim=1)

        q_types = torch.zeros(B, query.size(1), dtype=torch.long, device=query.device)
        kv_types = torch.cat([
            torch.ones(B, actives.size(1), device=query.device),
            torch.full((B, inactives.size(1)), 2, device=query.device)
        ], dim=1).long()

        query = self.embed(query, q_types)
        kv = self.embed(kv, kv_types)

        q, kv = self.block(query, kv, kv_mask, n_actives=n_actives)

        a = kv[:, :n_actives]
        i = kv[:, n_actives:]

        q = q.view(*query_shape[:-1], -1)

        return q, a, i

Test

In [19]:
cfg = SimpleNamespace(
    model=SimpleNamespace(
        transformer=SimpleNamespace(activity_embedding_dim=16)
    ),
    system=SimpleNamespace(
        ressources=SimpleNamespace(device="cpu")
    )
)

B, Q, A, I, D = 2, 1, 3, 2, 8

query_embedding = torch.randn(B, Q, D)
support_actives_embedding = torch.randn(B, A, D)
support_inactives_embedding = torch.randn(B, I, D)

support_actives_mask = torch.zeros(B, A)
support_inactives_mask = torch.zeros(B, I)

model = CrossAttentionModule(cfg)

q_out, a_out, i_out = model(
    query_embedding,
    support_actives_embedding,
    support_inactives_embedding,
    support_actives_mask,
    support_inactives_mask
)

print("Query:", q_out.shape)
print("Active:", a_out.shape)
print("Inactive:", i_out.shape)

with torch.no_grad():
    q_before = model.input_proj(query_embedding).clone()  # projected to 4 dim

    q_after, _, _ = model(
        query_embedding,
        support_actives_embedding,
        support_inactives_embedding,
        support_actives_mask,
        support_inactives_mask
    )

print("Query change magnitude:", torch.norm(q_after - q_before).item())

support_actives_embedding[:] = 5.0
support_inactives_embedding[:] = -5.0

q_pos, _, _ = model(
    query_embedding,
    support_actives_embedding,
    support_inactives_embedding,
    support_actives_mask,
    support_inactives_mask
)

support_actives_embedding[:] = -5.0
support_inactives_embedding[:] = 5.0

q_neg, _, _ = model(
    query_embedding,
    support_actives_embedding,
    support_inactives_embedding,
    support_actives_mask,
    support_inactives_mask
)

print("Support flip effect:", torch.norm(q_pos - q_neg).item())

full_mask_A = torch.ones_like(support_actives_mask)
full_mask_I = torch.ones_like(support_inactives_mask)

with torch.no_grad():
    q_before_proj = model.input_proj(query_embedding).clone()  # shape [2,1,4]

    q_masked, _, _ = model(
        query_embedding,
        support_actives_embedding,
        support_inactives_embedding,
        full_mask_A,
        full_mask_I
    )

print("Masked query change:", torch.norm(q_masked - q_before_proj).item())

query_embedding.requires_grad_(True)

q_out, _, _ = model(
    query_embedding,
    support_actives_embedding,
    support_inactives_embedding,
    support_actives_mask,
    support_inactives_mask
)

loss = q_out.sum()
loss.backward()

print("Gradient norm:", query_embedding.grad.norm().item())


Query: torch.Size([2, 1, 16])
Active: torch.Size([2, 3, 16])
Inactive: torch.Size([2, 2, 16])
Query change magnitude: 4.85295295715332
Support flip effect: 6.082489013671875
Masked query change: 5.9517951011657715
Gradient norm: 5.164990425109863


In [20]:
print(
    "attn_gate:",
    torch.sigmoid(model.block.gate_attn).item(),
    "ffn_gate:",
    torch.sigmoid(model.block.gate_ffn).item()
)
support_actives_embedding.zero_()
support_inactives_embedding.zero_()

q_zero, _, _ = model(
    query_embedding,
    support_actives_embedding,
    support_inactives_embedding,
    support_actives_mask,
    support_inactives_mask
)

print("Zero-support effect:",
      torch.norm(q_zero - q_before).item())

full_mask_A = torch.ones_like(support_actives_mask)
full_mask_I = torch.ones_like(support_inactives_mask)

q_masked, _, _ = model(
    query_embedding,
    support_actives_embedding,
    support_inactives_embedding,
    full_mask_A,
    full_mask_I
)

print("Fully masked attention:",
      torch.norm(q_masked - q_before).item())


attn_gate: 0.5744425058364868 ffn_gate: 0.6681877374649048
Zero-support effect: 5.1958723068237305
Fully masked attention: 5.223992347717285


In [30]:
with torch.no_grad():
    model.block.gate_attn.fill_(-20.0)  # sigmoid ≈ 0

q_no_attn, _, _ = model(
    query_embedding,
    support_actives_embedding,
    support_inactives_embedding,
    support_actives_mask,
    support_inactives_mask
)

print("No-attn change:",
      torch.norm(q_no_attn - q_before).item())

No-attn change: 4.824621200561523


In [26]:
# Project supports the same way the model does
with torch.no_grad():
    support_actives_proj = model.input_proj(support_actives_embedding)

with torch.no_grad():
    model.block.gate_ffn.fill_(1.0)

_, a2, _ = model(
    query_embedding,
    support_actives_embedding,
    support_inactives_embedding,
    support_actives_mask,
    support_inactives_mask
)

print(
    "Support update magnitude (enabled):",
    torch.norm(a2 - support_actives_proj).item()
)

Support update magnitude (enabled): 11.35517406463623


In [28]:
with torch.no_grad():
    model.block.gate_ffn.fill_(-20.0)  # disable support updates

_, a1, _ = model(
    query_embedding,
    support_actives_embedding,
    support_inactives_embedding,
    support_actives_mask,
    support_inactives_mask
)

print(
    "Support update magnitude:",
    torch.norm(a1 - support_actives_proj).item()
)


Support update magnitude: 11.765869140625


Older Results (I always add update):

Query: torch.Size([2, 1, 8])
Active: torch.Size([2, 3, 8])
Inactive: torch.Size([2, 2, 8])
Query change magnitude: 7.000400543212891
Support flip effect: 0.5861301422119141
Masked query change: 5.8696675300598145
Gradient norm: 8.994880676269531

With Token and Positional embeddings
Query: torch.Size([2, 1, 4])
Active: torch.Size([2, 3, 4]) 
Inactive: torch.Size([2, 2, 4]) 
Query change magnitude: 3.6029202938079834 
Support flip effect: 1.063355565071106
Masked query change: 3.9299356937408447 
Gradient norm: 3.32881760597229

With MoE
Query: torch.Size([2, 1, 4])
Active: torch.Size([2, 3, 4])
Inactive: torch.Size([2, 2, 4])
Query change magnitude: 4.143762588500977
Support flip effect: 0.7487077713012695
Masked query change: 6.8237624168396
Gradient norm: 2.7085113525390625

With swiglu
Query: torch.Size([2, 1, 4])
Active: torch.Size([2, 3, 4])
Inactive: torch.Size([2, 2, 4])
Query change magnitude: 2.5905020236968994
Support flip effect: 0.43406233191490173
Masked query change: 4.11211633682251
Gradient norm: 2.7517638206481934

With Scaled Dot-Product Attention
Query: torch.Size([2, 1, 4])
Active: torch.Size([2, 3, 4])
Inactive: torch.Size([2, 2, 4])
Query change magnitude: 5.255284786224365
Support flip effect: 1.4442298412322998
Masked query change: 5.317414283752441
Gradient norm: 4.158907890319824

With Rotary Positional Embedding
Query: torch.Size([2, 1, 4])
Active: torch.Size([2, 3, 4])
Inactive: torch.Size([2, 2, 4])
Query change magnitude: 3.630035638809204
Support flip effect: 1.257973551750183
Masked query change: 4.224307537078857
Gradient norm: 3.1558430194854736

With RMSNorm
Query: torch.Size([2, 1, 4])
Active: torch.Size([2, 3, 4])
Inactive: torch.Size([2, 2, 4])
Query change magnitude: 6.1455464363098145
Support flip effect: 0.760451078414917
Masked query change: 7.632925510406494
Gradient norm: 2.805803060531616

1. Convert Self-Attention → True Cross-Attention (Q ≠ K,V) ✅
2. Replace Manual Attention with scaled_dot_product_attention ✅
3. Make MoE Routing Soft (but keep Top-1 at inference) ✅
4. Add Attention Dropout (low risk, high stability) ✅
5. Split Token Type Embeddings into Active / Inactive Bias ✅
6. Residual Scaling (DeepNorm-lite) ✅
7. Optional: Gated Residual (advanced, optional) ✅
8. Optional advanced tweaks ✅

### Original code cross_attention code part

In [11]:
class CrossAttentionModule(nn.Module):
    
    def __init__(self, cfg: OmegaConf):
        
        super(CrossAttentionModule, self).__init__()

        self.cfg = cfg
        
        cfg_gpt = self.GPTConfig()
        self.transformer_block = self.TranformerBlock(cfg_gpt)

        # Initialization
        encoder_initialization = partial(init_weights, 'relu')
        self.apply(encoder_initialization)
        
    def forward(
        self,
        query_embedding: torch.Tensor,
        support_actives_embedding: torch.Tensor,
        support_inactives_embedding: torch.Tensor,
        support_actives_mask: torch.Tensor,
        support_inactives_mask: torch.Tensor,
    ) -> tuple:

        # Embedding dim of query and support set molecules
        embedding_dim = support_actives_embedding.shape[2]
        
        query_embedding = torch.cat(
            [
                query_embedding,
                torch.zeros_like(
                    query_embedding[
                        :, :, : self.cfg.model.transformer.activity_embedding_dim
                    ]
                ),
            ],
            dim=2,
        )

        support_actives_embedding = torch.cat(
            [
                support_actives_embedding,
                torch.ones_like(
                    support_actives_embedding[
                        :, :, : self.cfg.model.transformer.activity_embedding_dim
                    ]
                ),
            ],
            dim=2,
        )

        support_inactives_embedding = torch.cat(
            [
                support_inactives_embedding,
                (-1.0)
                * torch.ones_like(
                    support_inactives_embedding[
                        :, :, : self.cfg.model.transformer.activity_embedding_dim
                    ]
                ),
            ],
            dim=2,
        )

        # Concatenate query and support set molecules
        s = torch.cat(
            [query_embedding, support_actives_embedding, support_inactives_embedding],
            dim=1,
        )

        # Create padding mask
        padding_mask = torch.cat(
            [
                torch.tensor([False] * support_actives_mask.shape[0])
                .reshape(-1, 1) # [batch-size, 1]
                .to(self.cfg.system.ressources.device),  # query molecules
                # .to("cpu"),  # query molecules
                support_actives_mask,  # active support set molecules
                support_inactives_mask,  # inactive support set molecules
            ],
            dim=1,
        ).float()

        # Run transformer and update representations
        s_h = self.transformer_block(s, padding_mask)
        s_updated = s + s_h

        # Split representations into query, active, and inactive support set molecules
        query_embedding = s_updated[:, 0, :embedding_dim]
        query_embedding = torch.unsqueeze(query_embedding, 1)
        support_actives_embedding = s_updated[
            :, 1 : (support_actives_embedding.shape[1] + 1), :embedding_dim
        ]
        support_inactives_embedding = s_updated[
            :, (support_actives_embedding.shape[1] + 1) :, :embedding_dim
        ]

        return query_embedding, support_actives_embedding, support_inactives_embedding

    class TranformerBlock(nn.Module):

        def __init__(self, config):
            super().__init__()
            self.ln_1 = self.LayerNorm(config.n_embd, bias=config.bias)
            self.attn = self.SelfAttention(config)
            self.ln_2 = self.LayerNorm(config.n_embd, bias=config.bias)
            self.mlp = self.MLP(config)

        def forward(self, x, attn_mask):
            x = x + self.attn(self.ln_1(x), attn_mask)
            x = x + self.mlp(self.ln_2(x))
            return x
    
        class LayerNorm(nn.Module):

            def __init__(self, ndim, bias):
                super().__init__()
                self.weight = nn.Parameter(torch.ones(ndim))
                self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

            def forward(self, input):
                return F.layer_norm(input, self.weight.shape, self.weight, self.bias,
                                    1e-5)
        
        class SelfAttention(nn.Module):

            def __init__(self, config):
                super().__init__()
                
                self.cfg = config
                
                # query, key, value projections
                self.q_proj = nn.Linear(config.n_embd, config.n_qk_proj*config.n_head,
                                        bias=config.bias)
                self.k_proj = nn.Linear(config.n_embd, config.n_qk_proj*config.n_head,
                                        bias=config.bias)
                self.v_proj = nn.Linear(config.n_embd, config.n_v_proj*config.n_head,
                                        bias=config.bias)
                
                # output projection
                self.c_proj = nn.Linear(config.n_v_proj*config.n_head, config.n_embd,
                                        bias=config.bias)
                
                # regularization
                self.attn_dropout = nn.Dropout(config.dropout)
                self.resid_dropout = nn.Dropout(config.dropout)
                self.n_head = config.n_head
                self.n_embd = config.n_embd
                self.dropout = config.dropout

            def forward(self, x, attn_mask):
                B, T, C = x.size()
                assert len(attn_mask.shape) == 2
                
                attn_mask = attn_mask.view(attn_mask.shape[0], 1, attn_mask.shape[1])
                attn_mask = attn_mask.expand(attn_mask.shape[0], attn_mask.shape[2],
                                            attn_mask.shape[2])                
                attn_bias = attn_mask.masked_fill(attn_mask == 1., float("-inf"))
                
                # eventually we have to expand the mask to the number of heads
                attn_mask = attn_mask.view(attn_mask.shape[0], 1,
                                        attn_mask.shape[1], attn_mask.shape[2])
                attn_mask = attn_mask.expand(attn_mask.shape[0], self.n_head,
                                            attn_mask.shape[2], attn_mask.shape[3])
                assert len(attn_mask.shape) == 4

                q = self.q_proj(x).view(B, T, self.n_head, self.cfg.n_qk_proj).transpose(1, 2) # (B, nh, T, hs)
                k = self.k_proj(x).view(B, T, self.n_head, self.cfg.n_qk_proj).transpose(1, 2) # (B, nh, T, hs)
                v = self.v_proj(x).view(B, T, self.n_head, self.cfg.n_v_proj).transpose(1, 2) # (B, nh, T, hs)

                # Calculate self-attentions
                att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
                
                att = att.masked_fill(attn_mask == 1., float('-inf'))
                
                # Activations
                att = F.softmax(att, dim=-1)
                att = self.attn_dropout(att)
                y = att @ v 
                y = y.transpose(1, 2).contiguous().view(y.shape[0],
                                                        y.shape[2],
                                                        -1)

                # output projection
                y = self.resid_dropout(self.c_proj(y))
                return y

        class MLP(nn.Module):

            def __init__(self, config):
                super().__init__()
                self.c_fc    = nn.Linear(config.n_embd, 567, bias=config.bias)
                self.relu    = nn.ReLU()
                self.c_proj  = nn.Linear(567, config.n_embd, bias=config.bias)
                self.dropout = nn.Dropout(config.dropout)
            
            def forward(self, x):
                x = self.c_fc(x)
                x = self.relu(x)
                x = self.c_proj(x)
                x = self.dropout(x)
                return x

In [ ]:
class ScaledDotProductCrossAttention(nn.Module):
    def __init__(self, config, attn_dropout=0.1, rope_offset=0.0, temp=1.0):
        super().__init__()
        self.n_head = config.n_head
        self.head_dim = config.head_dim
        self.attn_dropout = attn_dropout
        self.rope_offset = rope_offset
        
        self.q_proj = nn.Linear(config.n_embd, config.n_embd)
        self.k_proj = nn.Linear(config.n_embd, config.n_embd)
        self.v_proj = nn.Linear(config.n_embd, config.n_embd)
        self.out_proj = nn.Linear(config.n_embd, config.n_embd)

        self.rotary = RotaryEmbedding(self.head_dim)
        
        self.active_bias = nn.Embedding(1, config.n_embd)
        self.inactive_bias = nn.Embedding(1, config.n_embd)
        
        # Learnable temperature
        self.log_temp = nn.Parameter(torch.log(torch.tensor(temp)))
        
    def forward(self, q, kv, kv_mask=None, n_actives=None):
        B, Tq, D = q.shape
        Tk = kv.size(1) 
        Na = n_actives
        assert Na is not None, "n_actives must be provided"
            
        # Project queries
        q = self.q_proj(q) 
            
        # Add active/inactive biases to supports
        kv = kv.clone()
        kv[:, :Na] += self.active_bias(torch.zeros(B, Na, dtype=torch.long, device=kv.device))
        kv[:, Na:] += self.inactive_bias(torch.zeros(B, Tk-Na, dtype=torch.long, device=kv.device)) 
            
        # Project keys & values 
        k = self.k_proj(kv) 
        sign = torch.cat([torch.ones(B, Na, 1, device=kv.device), -torch.ones(B, Tk-Na, 1, device=kv.device)], dim=1) 
        v = self.v_proj(kv) + sign * torch.cat([ self.active_bias(torch.zeros(B, Na, dtype=torch.long, device=kv.device)), 
                                                self.inactive_bias(torch.zeros(B, Tk-Na, dtype=torch.long, device=kv.device)) ], dim=1) 
            
        # Split heads 
        q = q.view(B, Tq, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, Tk, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, Tk, self.n_head, self.head_dim).transpose(1, 2)
            
        # Apply RoPE
        q = self.rotary(q, offset=self.rope_offset) 
        k = self.rotary(k, offset=0.0) 
            
        # Split active/inactive supports 
        k_a, k_i = k[:, :, :Na], k[:, :, Na:] 
        v_a, v_i = v[:, :, :Na], v[:, :, Na:] 
            
        mask_a = kv_mask[:, :Na] if kv_mask is not None else None 
        mask_i = kv_mask[:, Na:] if kv_mask is not None else None 
            
        if mask_a is not None:
            mask_a = mask_a[:, None, :] 
            mask_i = mask_i[:, None, :] 
            
        temp = torch.exp(self.log_temp) # ALWAYS positive 
            
        out_a = F.scaled_dot_product_attention( q / temp, k_a, v_a, attn_mask=mask_a, dropout_p=self.attn_dropout, is_causal=False )
            
        out_i = F.scaled_dot_product_attention( q / temp, k_i, v_i, attn_mask=mask_i, dropout_p=self.attn_dropout, is_causal=False ) 
            
        # --- FIX: normalize by number of actives/inactives --- 
        Na = k_a.size(2) 
        Ni = k_i.size(2)
        out = out_a / math.sqrt(Na) - out_i / math.sqrt(Ni) 
            
        out = out.transpose(1, 2).contiguous().reshape(B, Tq, D) 
        return self.out_proj(out)
    

    # -----------------------------
# TransformerBlock
# -----------------------------
class TransformerBlock(nn.Module):
    def __init__(self, config, attn_dropout=0.1, update_supports=True):
        super().__init__() 
        self.q_norm = RMSNorm(config.n_embd) 
        self.kv_norm_a = RMSNorm(config.n_embd) # for active supports
        self.kv_norm_i = RMSNorm(config.n_embd) # for inactive supports
        
        self.cross_attn = ScaledDotProductCrossAttention(config, attn_dropout=attn_dropout, rope_offset=0.1)
        
        self.ffn_norm = RMSNorm(config.n_embd) 
        self.moe = MoE(config.n_embd) 
        
        # Gated residuals (optional) 
        #self.gate_attn = nn.Parameter(torch.tensor(RESIDUAL_SCALE)) # learnable gate for attention 
        #self.gate_ffn = nn.Parameter(torch.tensor(RESIDUAL_SCALE)) # learnable gate for MoE / feedforward 

        # Asymmetric residual gates (learnable, with prior)
        self.gate_attn = nn.Parameter(torch.tensor(0.3))  # attention = conservative
        self.gate_ffn  = nn.Parameter(torch.tensor(0.7))  # MoE / FFN = stronger

        
        # Research switch: freeze supports 
        self.update_supports = update_supports 
        
    def forward(self, q, kv, kv_mask, n_actives):
        # Cross-attention: update query only 
        # Compute raw attention difference 
        Na = n_actives 
        kv_a = self.kv_norm_a(kv[:, :Na]) 
        kv_i = self.kv_norm_i(kv[:, Na:]) 
            
        kv_combined = torch.cat([kv_a, kv_i], dim=1) 
            
        delta = self.cross_attn( self.q_norm(q), kv_combined, kv_mask, n_actives=Na ) 
            
        # Normalize the delta
        delta_norm = RMSNorm(delta.size(-1))(delta)
            
        # Apply gated residual 
        #q = q + self.gate_attn * delta_norm
        attn_gate = torch.sigmoid(self.gate_attn)
        q = q + attn_gate * delta_norm

            
        # soft routing during training, optional support update 
        soft = self.training 
        if self.update_supports:
            #kv = kv + self.gate_ffn * self.moe(
            #    self.ffn_norm(kv),
            #    query=q, 
            #    soft=soft, 
            #    support_count=kv.size(1) # total supports 
            #    )
            ffn_gate = torch.sigmoid(self.gate_ffn)

            kv = kv + ffn_gate * self.moe(
                self.ffn_norm(kv),
                query=q,
                soft=soft,
                support_count=kv.size(1)
            )
 
         
        # else: kv is frozen (supports unchanged) 
            
        return q, kv

### With Rope

In [189]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from types import SimpleNamespace

# -----------------------------
# Utilities
# -----------------------------
def init_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

# -----------------------------
# GPT-like config
# -----------------------------
class GPTConfig:
    def __init__(self, n_embd, n_head=2):
        self.n_embd = n_embd
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        assert self.head_dim * n_head == n_embd

# -----------------------------
# RMSNorm
# -----------------------------
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(rms + self.eps)
        return x * self.weight

# -----------------------------
# Rotary Positional Embedding
# -----------------------------
class RotaryEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        assert dim % 2 == 0, "Head dim must be even for RoPE"

    def forward(self, x, offset=0.0):
        """
        x: [B, H, T, D]
        """
        B, H, T, D = x.shape

        pos = torch.arange(T, device=x.device, dtype=torch.float32) + offset
        dim_range = torch.arange(0, D, 2, device=x.device, dtype=torch.float32)
        inv_freq = 1.0 / (10000 ** (dim_range / D))
        angles = torch.einsum("i,j->ij", pos, inv_freq)

        cos = angles.cos().unsqueeze(0).unsqueeze(0)
        sin = angles.sin().unsqueeze(0).unsqueeze(0)

        x_even = x[..., ::2]
        x_odd = x[..., 1::2]

        x_rot_even = x_even * cos - x_odd * sin
        x_rot_odd = x_even * sin + x_odd * cos

        x_out = torch.zeros_like(x)
        x_out[..., ::2] = x_rot_even
        x_out[..., 1::2] = x_rot_odd
        return x_out

# -----------------------------
# Input embedding (type embedding)
# -----------------------------
class InputEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.type_emb = nn.Embedding(3, dim)

    def forward(self, x, types):
        return x + self.type_emb(types)

# -----------------------------
# SwiGLU
# -----------------------------
class SwiGLU(nn.Module):
    def forward(self, x):
        x, gate = x.chunk(2, dim=-1)
        return x * F.silu(gate)

# -----------------------------
# Mixture-of-Experts
# -----------------------------
class MoE(nn.Module):
    def __init__(self, d_model, n_experts=4, hidden_dim=32, dropout=0.05):
        super().__init__()
        self.router = nn.Linear(d_model, n_experts)
        self.n_experts = n_experts

        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, hidden_dim * 2),
                SwiGLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, d_model),
            ) for _ in range(n_experts)
        ])

    def forward(self, x, query=None, soft=True, support_count=None):
        B, T, D = x.shape

        n_experts_use = self.n_experts
        if support_count is not None:
            n_experts_use = min(self.n_experts, max(1, support_count // 2))

        experts = self.experts[:n_experts_use]

        logits = self.router(x)[:, :, :n_experts_use]

        if query is not None:
            q_vec = query.mean(dim=1, keepdim=True)
            sim = torch.einsum("bqd,btd->bqt", q_vec, x).squeeze(1)
            logits = logits + sim.unsqueeze(-1)

        probs = F.softmax(logits, dim=-1)

        if soft:
            out = sum(expert(x) * probs[..., i:i+1]
                      for i, expert in enumerate(experts))
        else:
            expert_id = logits.argmax(dim=-1)
            out = torch.zeros_like(x)
            for i, expert in enumerate(experts):
                mask = expert_id == i
                if mask.any():
                    out[mask] = expert(x[mask])

        return out

# -----------------------------
# Cross Attention
# -----------------------------
class ScaledDotProductCrossAttention(nn.Module):
    def __init__(self, config, attn_dropout=0.1, rope_offset=0.0, temp=1.0):
        super().__init__()

        self.n_head = config.n_head
        self.head_dim = config.head_dim
        self.attn_dropout = attn_dropout
        self.rope_offset = rope_offset

        self.q_proj = nn.Linear(config.n_embd, config.n_embd)
        self.k_proj = nn.Linear(config.n_embd, config.n_embd)
        self.v_proj = nn.Linear(config.n_embd, config.n_embd)
        self.out_proj = nn.Linear(config.n_embd, config.n_embd)

        self.rotary = RotaryEmbedding(self.head_dim)

        self.active_bias = nn.Embedding(1, config.n_embd)
        self.inactive_bias = nn.Embedding(1, config.n_embd)

        self.log_temp = nn.Parameter(torch.log(torch.tensor(temp)))

    def forward(self, q, kv, kv_mask=None, n_actives=None):
        B, Tq, D = q.shape
        Tk = kv.size(1)
        Na = n_actives
        Ni = Tk - Na

        q = self.q_proj(q)

        kv = kv.clone()
        kv[:, :Na] += self.active_bias.weight
        kv[:, Na:] += self.inactive_bias.weight

        k = self.k_proj(kv)
        v = self.v_proj(kv)

        q = q.view(B, Tq, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, Tk, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, Tk, self.n_head, self.head_dim).transpose(1, 2)

        q = self.rotary(q, offset=self.rope_offset)
        k = self.rotary(k)

        k_a, k_i = k[:, :, :Na], k[:, :, Na:]
        v_a, v_i = v[:, :, :Na], v[:, :, Na:]

        temp = torch.exp(self.log_temp)

        out = 0

        if Na > 0:
            out_a = F.scaled_dot_product_attention(q / temp, k_a, v_a, dropout_p=self.attn_dropout)
            out += out_a / math.sqrt(Na)

        if Ni > 0:
            out_i = F.scaled_dot_product_attention(q / temp, k_i, v_i, dropout_p=self.attn_dropout)
            out -= out_i / math.sqrt(Ni)

        out = out.transpose(1, 2).contiguous().reshape(B, Tq, D)
        return self.out_proj(out)

# -----------------------------
# Transformer block
# -----------------------------
class TransformerBlock(nn.Module):
    def __init__(self, config, attn_dropout=0.1, update_supports=True):
        super().__init__()

        self.q_norm = RMSNorm(config.n_embd)
        self.kv_norm_a = RMSNorm(config.n_embd)
        self.kv_norm_i = RMSNorm(config.n_embd)

        self.cross_attn = ScaledDotProductCrossAttention(config, attn_dropout)
        self.delta_norm = RMSNorm(config.n_embd)

        self.ffn_norm = RMSNorm(config.n_embd)
        self.moe = MoE(config.n_embd)

        self.gate_attn = nn.Parameter(torch.tensor(0.3))
        self.gate_ffn = nn.Parameter(torch.tensor(0.7))

        self.update_supports = update_supports

    def forward(self, q, kv, kv_mask, n_actives):
        Na = n_actives

        kv_a = self.kv_norm_a(kv[:, :Na])
        kv_i = self.kv_norm_i(kv[:, Na:])
        kv_combined = torch.cat([kv_a, kv_i], dim=1)

        delta = self.cross_attn(self.q_norm(q), kv_combined, kv_mask, n_actives=Na)
        delta = self.delta_norm(delta)

        q = q + torch.sigmoid(self.gate_attn) * delta

        if self.update_supports:
            kv = kv + torch.sigmoid(self.gate_ffn) * self.moe(
                self.ffn_norm(kv),
                query=q,
                soft=self.training,
                support_count=kv.size(1)
            )

        return q, kv

# -----------------------------
# CrossAttentionModule
# -----------------------------
class CrossAttentionModule(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.model_dim = cfg.model.transformer.activity_embedding_dim
        config = GPTConfig(n_embd=self.model_dim)

        self.input_proj = nn.Linear(8, self.model_dim)
        self.embed = InputEmbedding(self.model_dim)
        self.block = TransformerBlock(config)

        self.apply(init_weights)

    def forward(self, query, actives, inactives, act_mask, inact_mask):
        B = query.size(0)
        D = query.size(-1)

        query_shape = query.shape

        query = query.view(B, -1, D)
        actives = actives.view(B, -1, D)
        inactives = inactives.view(B, -1, D)

        query = self.input_proj(query)
        actives = self.input_proj(actives)
        inactives = self.input_proj(inactives)

        n_actives = actives.size(1)

        kv = torch.cat([actives, inactives], dim=1)
        kv_mask = torch.cat([act_mask.view(B, -1), inact_mask.view(B, -1)], dim=1)

        q_types = torch.zeros(B, query.size(1), dtype=torch.long, device=query.device)
        kv_types = torch.cat([
            torch.ones(B, actives.size(1), device=query.device),
            torch.full((B, inactives.size(1)), 2, device=query.device)
        ], dim=1).long()

        query = self.embed(query, q_types)
        kv = self.embed(kv, kv_types)

        q, kv = self.block(query, kv, kv_mask, n_actives=n_actives)

        a = kv[:, :n_actives]
        i = kv[:, n_actives:]

        q = q.view(*query_shape[:-1], -1)

        return q, a, i


# -----------------------------
# TEST
# -----------------------------
cfg = SimpleNamespace(
    model=SimpleNamespace(
        transformer=SimpleNamespace(activity_embedding_dim=16)
    ),
    system=SimpleNamespace(
        ressources=SimpleNamespace(device="cpu")
    )
)

B, Q, A, I, D = 2, 1, 3, 2, 8

query_embedding = torch.randn(B, Q, D)
support_actives_embedding = torch.randn(B, A, D)
support_inactives_embedding = torch.randn(B, I, D)

support_actives_mask = torch.zeros(B, A)
support_inactives_mask = torch.zeros(B, I)

model = CrossAttentionModule(cfg)

q_out, a_out, i_out = model(
    query_embedding,
    support_actives_embedding,
    support_inactives_embedding,
    support_actives_mask,
    support_inactives_mask
)

print("Query:", q_out.shape)
print("Active:", a_out.shape)
print("Inactive:", i_out.shape)

with torch.no_grad():
    q_before = model.input_proj(query_embedding).clone()  # projected to 4 dim

    q_after, _, _ = model(
        query_embedding,
        support_actives_embedding,
        support_inactives_embedding,
        support_actives_mask,
        support_inactives_mask
    )

print("Query change magnitude:", torch.norm(q_after - q_before).item())

support_actives_embedding[:] = 5.0
support_inactives_embedding[:] = -5.0

q_pos, _, _ = model(
    query_embedding,
    support_actives_embedding,
    support_inactives_embedding,
    support_actives_mask,
    support_inactives_mask
)

support_actives_embedding[:] = -5.0
support_inactives_embedding[:] = 5.0

q_neg, _, _ = model(
    query_embedding,
    support_actives_embedding,
    support_inactives_embedding,
    support_actives_mask,
    support_inactives_mask
)

print("Support flip effect:", torch.norm(q_pos - q_neg).item())

full_mask_A = torch.ones_like(support_actives_mask)
full_mask_I = torch.ones_like(support_inactives_mask)

with torch.no_grad():
    q_before_proj = model.input_proj(query_embedding).clone()  # shape [2,1,4]

    q_masked, _, _ = model(
        query_embedding,
        support_actives_embedding,
        support_inactives_embedding,
        full_mask_A,
        full_mask_I
    )

print("Masked query change:", torch.norm(q_masked - q_before_proj).item())

query_embedding.requires_grad_(True)

q_out, _, _ = model(
    query_embedding,
    support_actives_embedding,
    support_inactives_embedding,
    support_actives_mask,
    support_inactives_mask
)

loss = q_out.sum()
loss.backward()

print("Gradient norm:", query_embedding.grad.norm().item())

Query: torch.Size([2, 1, 16])
Active: torch.Size([2, 3, 16])
Inactive: torch.Size([2, 2, 16])
Query change magnitude: 7.138394355773926
Support flip effect: 4.260286331176758
Masked query change: 6.95396614074707
Gradient norm: 4.295406341552734
